# Mehrperiodisches Forecasting für dynamische Entscheidungsfindung – SafeRunner als Schnellstart

Dieses Notebook zeigt:
- Wie man Experimente stabil über `safe_runner.run_experiment` startet
- Was die verfügbaren Modelle und Wrapper sind
- Was die wichtigsten Parameter bedeuten

## 0. Modelle & Wrapper – Kurzüberblick

### Wrapper

- ``direct`` (One-model-per-horizon)
  - Prinzip: Für jeden Horizont h wird ein eigenes Regressionsmodell trainiert (z. B. 4 Modelle für H=1,7,14,28).
  - Pro: Unabhängige Spezialisierung je Horizont; oft robuste Performanz.
  - Contra: Mehr Trainingszeit/Speicher; ignoriert Korrelationen zwischen Horizonten.
- ``mimo`` (Multi-output, gemeinsames Modell)
  - Prinzip: Ein Modell sagt alle Horizonte gleichzeitig voraus (Multi-Target-Regression).
  - Pro: Nutzt Querverbindungen zwischen Horizonten; HPO ist hier freigeschaltet.
  - Contra: Wenn ein Horizont schwerer lernbar ist, kann er die anderen „mitziehen“.

Regel: ``run_hpo=True`` ist nur mit ``wrapper="mimo"`` erlaubt.

### Modelle

#### LightGBM (`lightgbm`) und XGBoost (`xgboost`)

##### ``lightgbm``
* **Typ:** Gradient Boosted Trees (Leaf-wise).
* **Wrapper:** `direct` **und** `mimo`.
* **Einsatz:** Starke tabellarische Baseline, schnell, gut mit vielen Features (auch kategorialen als codes).
* **Wichtig:** GPU geht nur mit **GPU-fähiger LightGBM-Build**.

**Key-Knobs (typisch):** `num_leaves`, `max_depth`, `n_estimators`, `learning_rate`, `subsample`, `colsample_bytree`, `max_bin`, `reg_lambda`.

---
##### ``xgboost``
* **Typ:** Gradient Boosted Trees (Histogram/Approx).
* **Wrapper:** `direct` **und** `mimo`.
* **Einsatz:** Sehr stabil; mit GPU oft schneller bei HPO.
* **Wichtig (GPU):**

  * XGBoost **< 2.0**: `tree_method="gpu_hist"`, `predictor="gpu_predictor"`, `gpu_id=0`.
  * XGBoost **≥ 2.0**: `device="cuda"` (kein `gpu_id/predictor` nötig).

**Key-Knobs (typisch):** `max_depth`, `min_child_weight`, `subsample`, `colsample_bytree`, `reg_lambda`, `reg_alpha`, `n_estimators`, `learning_rate`.

---

#### N-HiTS (`nhits`) und TFT – Temporal Fusion Transformer (`tft`)

##### ``nhits``
* **Typ:** Deep Learning, **globales** Modell (lernt über **alle Serien**).
* **Architektur (vereinfacht):** Mehrstufige **Backcast/Forecast**-Blöcke mit mehrskaligen Filterungen; stark bei **langfristigen** Mustern.
* **Wrapper:** Semantisch wie `mimo` (gibt alle Horizonte auf einmal aus).
* **GPU:** Nutzt CUDA automatisch (falls verfügbar).
* **Key-Knobs (typisch):** `input_chunk_length`, `n_epochs`, `batch_size`, `dropout`.
---
##### ``tft``
* **Typ:** Deep Learning, **global**; kombiniert **LSTM** + **Multi-Head Attention** + **Gating** + **Variable Selection**.
* **Stärken:** Umgang mit **statischen & dynamischen** Kovariaten, **Interpretierbarkeit**.
* **Wrapper:** Semantisch wie `mimo` (Multi-Horizon).
* **GPU:** Nutzt CUDA automatisch (falls verfügbar).
* **Key-Knobs (typisch):** `input_chunk_length`, `hidden_size`, `lstm_layers`, `num_attention_heads`, `dropout`, `n_epochs`, `batch_size`.

---

#### GNS – Global Naive Seasonal (`gns`)

* **Typ:** Saisonaler **Naive-Baseline**-Ansatz, global über alle Serien angewandt.
* **Idee:** Vorhersage = **Durchschnitt** der Werte an **derselben saisonalen Position** in den letzten `K` Saisons.
* **Parameter `K`:**

  * `K=1` ⇒ **Seasonal Naive** (nur letzte Saisonposition).
  * `K>1` ⇒ **Geglättet** (mittelt über `K` Saisonzyklen) → robust gegen Rauschen, reagiert aber **träge** auf Trends/Shifts.
* **Saisonperiode `m` (z. B. 7):** Definiert den **Abstand** zwischen gleichen Saisonpositionen (z. B. Wochentag).
* **Wrapper:** Nicht relevant (Vorhersage aller Horizonte via saisonalem Muster).

> **Faustregel:** `K` in **\[1..28]** testen (wochenbasiert). Kleine `K` → reaktiver, große `K` → glatter.

---


## 1. Setup & Hilfe

Führen die nächste Zelle aus um eine detalierte Anleitung zu bekommen!

In [ ]:
from safe_runner import explain_params
print(explain_params())

## 2. Ausgeführte Runs

### Übersicht untersuchter Runs (Modelle)


| Nr. | Modell | hpTuned | Dataset | Wrapper  | HPO   | Log (W&B/CSV) |
|----:|--------|--------|---------|----------|-------|---------------|
| **LightGBM – Basic** |||||||
| 1  | lgbm   | ❌ | m5     | mimo    | ❌ | ✅ |
| 2  | lgbm   | ❌ | m5     | direct  | ❌ | ✅ |
| 3  | lgbm   | ❌ | bakery | mimo    | ❌ | ✅ |
| 4  | lgbm   | ❌ | bakery | direct  | ❌ | ✅ |
| **LightGBM – HPO Runs** |||||||
| 5  | lgbm   | ❌ | m5     | mimo    | ✅ | ❌ |
| 6  | lgbm   | ❌ | bakery | mimo    | ✅ | ❌ |
| **LightGBM – Tuned Params** |||||||
| 7  | lgbm   | ✅ | m5     | mimo    | ❌ | ✅ |
| 8  | lgbm   | ✅ | m5     | direct  | ❌ | ✅ |
| 9  | lgbm   | ✅ | bakery | mimo    | ❌ | ✅ |
| 10 | lgbm   | ✅ | bakery | direct  | ❌ | ✅ |
| **XGBoost – Basic** |||||||
| 11 | xgb    | ❌ | m5     | mimo    | ❌ | ✅ |
| 12 | xgb    | ❌ | m5     | direct  | ❌ | ✅ |
| 13 | xgb    | ❌ | bakery | mimo    | ❌ | ✅ |
| 14 | xgb    | ❌ | bakery | direct  | ❌ | ✅ |
| **XGBoost – HPO Runs** |||||||
| 15 | xgb    | ❌ | m5     | mimo    | ✅ | ❌ |
| 16 | xgb    | ❌ | bakery | mimo    | ✅ | ❌ |
| **XGBoost – Tuned Params** |||||||
| 17 | xgb    | ✅ | m5     | mimo    | ❌ | ✅ |
| 18 | xgb    | ✅ | m5     | direct  | ❌ | ✅ |
| 19 | xgb    | ✅ | bakery | mimo    | ❌ | ✅ |
| 20 | xgb    | ✅ | bakery | direct  | ❌ | ✅ |
| **Deep Learning** |||||||
| 21 | nhits  | ❌ | m5     | native  | ❌ | ✅ |
| 22 | nhits  | ❌ | bakery | native  | ❌ | ✅ |
| 23 | tft    | ❌ | m5     | native  | ❌ | ✅ |
| 24 | tft    | ❌ | bakery | native  | ❌ | ✅ |
| **Baselines** |||||||
| 25 | gns_k7 | ❌ | m5     | baseline| ❌ | ✅ |
| 26 | gns_k7 | ❌ | bakery | baseline| ❌ | ✅ |



### Haupt-Parameter

| Parameter         | Wirkung                                                                                   | Werte                                                     | Bsp. Run_Name                     |
|-------------------|-------------------------------------------------------------------------------------------|-----------------------------------------------------------|-----------------------------------|
| MODEL             | wählt Model aus                                                                           | ``xgboost``,``lightgbm``, <br>``nhits``, ``tft``, ``gns`` | z.B. "xgb-...."                   |
| USE_HPTUNE_PARAMS | wählt welche Params Model verwenden soll                                                  | ``True`` oder ``False``                                     | z.B  "xgb-basic-..."              |
| WRAPPER           | für xgb, lgbm Wrapper wählbar, sont egal                                                  | ``mimo`` oder ``direct``                                    | z.B. "xgb-basic-mimo-..."         |
| DATASET           | entscheiden welcher Datenstaz verwendet wird                                              | ``m5`` oder ``bakery``                                      | z.B. "xgb-basic-mimo-m5..."       |
| RUN_SUFFIX        | frei wählbarer Sting für Run Name                                                         | nix oder beliebig                                         | z.B. "xgb-basic-mimo-m5_subset"   |
| SUBSET            | entscheiden ob 4 Serien (2 Stores x 2 Items)   <br>oder ganzes Dataset verwendet wird     | ``True`` oder ``False``                                     |                                   |
| RUN_HPO           | ob HyperParameterTuning beste Params suchen soll                                          | ``True`` oder ``False``                                     |                                   |
| optional:<br>save_csv_out_eval | Speichert gleich den Run in die run_summary.csv, alternative siehe unten.    | nix oder ``True`` oder ``False``  ||

Beispiel für einen Run:
result = run_experiment( run_suffix="`test_1`", 
                    model="`xgboost`", 
                    wrapper="`mimo`", 
                    dataset="`bakery`",
                    use_wandb=`False`, 
                    use_hptuned_params=`False`, 
                    run_hpo=`False`, 
                    subset=`True`,)

- Füge wie folgt einen abgeschlossenen run zur run_summary hinzu: append_log_payload(result[``"_log_payload"``])

### Ausgeführte Runs 

- die in run_summary.csv gespeichert wurden und
- auf wandb mit den Suffex `full_FIN` gespeichert wurden

In [ ]:
# importiernen der notwendigen Module
from safe_runner import run_experiment
from results_logger import append_log_payload

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",
    category=UserWarning,
)

In [ ]:
# # xgboost basic runs
# res_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_xgb_direct_m5 = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="m5", wrapper="direct", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_xgb_mimo_bakery = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_xgb_direct_bakery = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="bakery", wrapper="direct", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# # HPO runs
# res_xgb_mimo_m5_hpo = run_experiment(run_suffix="HPO", model="xgboost", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=True, subset=False,)  
# res_xgb_mimo_bakery_hpo = run_experiment(run_suffix="HPO", model="xgboost", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=True, subset=False,)  
# # final runs with best params from HPO
# res_xgb_mimo_m5_tuned = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_xgb_direct_m5_tuned = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="m5", wrapper="direct", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_xgb_mimo_bakery_tuned = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_xgb_direct_bakery_tuned = run_experiment(run_suffix="full_FIN", model="xgboost", dataset="bakery", wrapper="direct", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)

In [ ]:
# # lightgbm basic runs
# res_lgbm_mimo_m5 = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_lgbm_direct_m5 = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="m5", wrapper="direct", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_lgbm_mimo_bakery = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# res_lgbm_direct_bakery = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="bakery", wrapper="direct", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# # HPO runs
# res_lgbm_mimo_m5_hpo = run_experiment(run_suffix="HPO", model="lightgbm", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=True, subset=False,)  
# res_lgbm_mimo_bakery_hpo = run_experiment(run_suffix="HPO", model="lightgbm", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=False, run_hpo=True, subset=False,)  
# # final runs with best params from HPO
# res_lgbm_mimo_m5_tuned = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="m5", wrapper="mimo", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_lgbm_direct_m5_tuned = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="m5", wrapper="direct", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_lgbm_mimo_bakery_tuned = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="bakery", wrapper="mimo", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  
# res_lgbm_direct_bakery_tuned = run_experiment(run_suffix="full_FIN", model="lightgbm", dataset="bakery", wrapper="direct", use_wandb=True, use_hptuned_params=True, run_hpo=False, subset=False,)  

In [ ]:
# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="tft", dataset="m5", wrapper="native", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="tft", dataset="bakery", wrapper="native", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  

# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="nhits", dataset="m5", wrapper="native", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  
# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="nhits", dataset="bakery", wrapper="native", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False,)  

# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="gns", dataset="m5", wrapper="baseline", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False, gns_k=7,)  
# un_xgb_mimo_m5 = run_experiment(run_suffix="full_FIN", model="gns", dataset="bakery", wrapper="baseline", use_wandb=True, use_hptuned_params=False, run_hpo=False, subset=False, gns_k=7,)  

In [ ]:
# append_log_payload(res_xgb_mimo_m5["_log_payload"])

## 3. Ergebnis

Hinweis die zwei NaN Werte bei WanDB_WRL kommen davon das verwessen wurde sie geloggten Runs auf Wandb auch in der runs_summary.csv zu speichern, deswegen wurde nachträglich identische Runs gemacht ohne wanDB und in die csv gespeichert. 


In [ ]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd().parents[1] if len(Path.cwd().parents) else Path.cwd()

EVAL_DIR = ROOT_DIR / "outputs/evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_SUMMARIES_PATH = EVAL_DIR / "runs_summary.csv"

path = r"outputs\evaluation\runs_summary.csv"

print("Evaluation dir:", EVAL_DIR.resolve())
print("Expecting run summaries at:", RUN_SUMMARIES_PATH.resolve())

df_summaries = pd.read_csv(RUN_SUMMARIES_PATH) if RUN_SUMMARIES_PATH.exists() else pd.DataFrame()
df_summaries.sort_values(by=["Model","Datensatz", "RMSSE_total"], inplace=True)
df_summaries

## 4. Proberie es selbst aus 

Denk dran die nötigen Bibliotheken installiert zu haben. Du kannst dafür einfach mit ``pip install -r requirements.txt`` alle nötigen auf einmal installieren.

- Schau dir gerne die mkdocs Seite an oder lade sie lokal mittels ``mkdocs serve`` im Terminal selber.
- Schau dir gerne die geloggten Runs auf wandb und den dazugehörigen Report an.

In [ ]:
# importiernen der notwendigen Module
from safe_runner import run_experiment
from results_logger import append_log_payload

import warnings
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMRegressor was fitted with feature names",
    category=UserWarning,
)

In [ ]:
res_test = run_experiment(
    run_suffix="probe_test",
    model="xgboost",
    wrapper="direct",
    dataset="m5",
    subset=True,
    use_wandb=False,
    use_hptuned_params=False,
    run_hpo=False,
    save_csv_out_eval=False,
)

print("RMSSE total:", res_test["rmsse_total"])

In [ ]:
print("RMSSE total:", res_test["rmsse_total"])
print("Wurden getunte Parameter verwendet?:", res_test["used_tuned_params"])
print("Genutztes Gerät (device):", res_test["params"].get("device"))
print("Learning Rate:", res_test["params"].get("learning_rate"))
print("Num Leaves:", res_test["params"].get("num_leaves"))
print("Anzahl Bäume (n_estimators):", res_test["params"].get("n_estimators"))
print("Parameterquelle:", res_test["param_source"])
print("Train CPU Systemzeit (Sekunden):", res_test["train_profile"]["train_cpu_time_system_s"])